In [1]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [3]:
# The notebook will need couple module files from code directory.
import os,sys
# os.path.join('..', 'code') is the relative path for code folder, module_path convert it to absolute path, final absolute url: d:\\Study\\Python\\llm-zoomcamp\\code
module_path = os.path.abspath(os.path.join('..', 'code'))

# append the code folder in sys.path if not already exists (100% likely)
if module_path not in sys.path:
    sys.path.append(module_path)

In [4]:
from rag_helper import RAGBase
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [5]:
instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [6]:
assistant.rag("How do I run Ollama locally?")

'To run Ollama locally:\n\n1. Install Ollama from **https://ollama.com/download** for your operating system:\n   - **macOS**: download the `.pkg`\n   - **Windows**: download the `.msi`\n   - **Linux**: run:\n     ```bash\n     curl -fsSL https://ollama.com/install.sh | sh\n     ```\n\n2. Open a terminal and start a model locally with:\n   ```bash\n   ollama run llama3\n   ```\n\n   This downloads the LLaMA 3 model, starts it locally, and opens a chat-like interface.\n\n3. To test that the local server is running, use:\n   ```bash\n   curl http://localhost:11434\n   ```\n\n   You should get a response like:\n   ```json\n   {"models": [...]}  \n   ```\n\nIf you want to use it from Python, install the client with:\n\n```bash\npip install ollama\n```'

In [7]:
assistant.rag("How do I run Olama locally?")

'I don’t see any context about running Olama locally.\n\nThe provided FAQ only covers things like MCP Inspector, token counting, API keys, table listing, and RAG evaluation. If you want, I can help with one of those topics from the context.'

In [8]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
)

response.output_text

'Absolutely — if enrollment is still open, you can usually join.\n\nTo confirm, I’d need one of these:\n- the course name\n- the platform or school offering it\n- the start date or link\n\nIf you want, I can help you figure out:\n1. whether you’re eligible,\n2. how to enroll,\n3. or draft a quick message to the instructor/admissions team.'

In [9]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [10]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [11]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output

[ResponseFunctionToolCall(arguments='{"query":"join course discovered late enroll late registration can I join"}', call_id='call_2jxpgVSx43jE0KHTIPs6e9vA', name='search', type='function_call', id='fc_0cbfabcb6605a407006a1e2dfb0fd48191a11c004162050b9f', namespace=None, status='completed')]

In [12]:
import json

call = response.output[0]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

In [13]:
messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

In [14]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output_text

'Yes — you can still join and start learning.\n\nIf you want a certificate, though, you need to submit your project while submissions are still open.'

In [15]:
usage = response.usage
usage.input_tokens, usage.output_tokens

def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(652, 33)
print("Total cost: $", round(result["total_cost"], 8))

Total cost: $ 0.0001176


In [16]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [17]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [18]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

messages.extend(response.output)
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

function_call: search {"query":"join the course late enroll discovered the course can I join FAQ"}


In [19]:
it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
ASSISTANT:
Yes — you can still join.

If you want a certificate, though, you need to submit your project while the course is still accepting submissions. If you’re just learning without aiming for a certificate, you can follow along in self-paced mode.

Would you like me to help with anything else about the course?


In [20]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [21]:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search {"query":"Olama run locally install local setup Ollama"}
iteration #2...
function_call: search {"query":"Ollama server localhost 11434 ollama serve restart connection refused local"}
iteration #3...
ASSISTANT:
To run Ollama locally:

1. **Install Ollama**
   - Go to: https://ollama.com/download
   - Choose your OS:
     - **macOS**: download and install the `.pkg`
     - **Windows**: download and install the `.msi`
     - **Linux**:
       ```bash
       curl -fsSL https://ollama.com/install.sh | sh
       ```

2. **Start a model locally**
   In a terminal, run:
   ```bash
   ollama run llama3
   ```
   This will download the model, start it locally, and open a chat prompt.

3. **Check that the local server is running**
   Ollama usually serves on `http://localhost:11434`.
   You can test it with:
   ```bash
   curl http://localhost:11434
   ```

4. **If you get a connection refused error**
   Restart the server with:
   ```bash
   ollama serve
   

'To run Ollama locally:\n\n1. **Install Ollama**\n   - Go to: https://ollama.com/download\n   - Choose your OS:\n     - **macOS**: download and install the `.pkg`\n     - **Windows**: download and install the `.msi`\n     - **Linux**:\n       ```bash\n       curl -fsSL https://ollama.com/install.sh | sh\n       ```\n\n2. **Start a model locally**\n   In a terminal, run:\n   ```bash\n   ollama run llama3\n   ```\n   This will download the model, start it locally, and open a chat prompt.\n\n3. **Check that the local server is running**\n   Ollama usually serves on `http://localhost:11434`.\n   You can test it with:\n   ```bash\n   curl http://localhost:11434\n   ```\n\n4. **If you get a connection refused error**\n   Restart the server with:\n   ```bash\n   ollama serve\n   ```\n   Or, in a notebook environment:\n   ```bash\n   !nohup ollama serve > nohup.out 2>&1 &\n   ```\n\n5. **Optional: use it from Python**\n   Install the client:\n   ```bash\n   pip install ollama\n   ```\n   Examp

In [22]:
agent_loop(instructions, "I just discovered the course. Can I still join it?")

iteration #1...
function_call: search {"query":"join course late enrollment discovered course can I still join"}
function_call: search {"query":"late enrollment course join after start FAQ"}
iteration #2...
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, make sure to submit your project while submissions are still open. You can also start learning and submitting homework as long as the forms are open.

If you’d like, I can also help with what to do first after joining or explain the certificate requirements.


'Yes — you can still join the course.\n\nIf you want a certificate, make sure to submit your project while submissions are still open. You can also start learning and submitting homework as long as the forms are open.\n\nIf you’d like, I can also help with what to do first after joining or explain the certificate requirements.'

In [23]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "I just discovered the course. Can I join it?")

iteration #1...
function_call: search {"query":"join course discovered late can I join course enrollment late FAQ"}
iteration #2...
function_call: search {"query":"certificate project submission while still accepting submissions peer review live cohort self-paced FAQ"}
iteration #3...
ASSISTANT:
Yes — you can still join the course.

If you want to receive a certificate, you’ll need to submit your project while submissions are still open. Also, certificates are only available if you finish with a live cohort, not in self-paced mode.

If you want, I can also help you figure out whether it’s still possible to get a certificate right now. Anything else you’d like to explore?


'Yes — you can still join the course.\n\nIf you want to receive a certificate, you’ll need to submit your project while submissions are still open. Also, certificates are only available if you finish with a live cohort, not in self-paced mode.\n\nIf you want, I can also help you figure out whether it’s still possible to get a certificate right now. Anything else you’d like to explore?'

In [24]:
agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit opening chess definition"}
iteration #2...
function_call: search {"query":"queen gambit chess opening moves white d4 d5 c4"}
iteration #3...
ASSISTANT:
The **Queen’s Gambit** is a classic **chess opening** that starts with:

1. **d4 d5**
2. **c4**

White offers the c-pawn to try to **pull Black’s d-pawn away** from the center and gain control of the board. Despite the name, it’s not really a “gambit” in the sense of a long-term sacrifice; Black can often keep the pawn or return it later.

A very common line is:
- **1. d4 d5 2. c4**

Main ideas:
- fight for the center
- open lines for White’s pieces
- create pressure on Black’s d5 pawn

There are a few major responses by Black, like:
- **Queen’s Gambit Accepted**: `2...dxc4`
- **Queen’s Gambit Declined**: `2...e6`
- **Slav Defense**: `2...c6`

If you want, I can also explain the difference between **Queen’s Gambit Accepted vs Declined**.


'The **Queen’s Gambit** is a classic **chess opening** that starts with:\n\n1. **d4 d5**\n2. **c4**\n\nWhite offers the c-pawn to try to **pull Black’s d-pawn away** from the center and gain control of the board. Despite the name, it’s not really a “gambit” in the sense of a long-term sacrifice; Black can often keep the pawn or return it later.\n\nA very common line is:\n- **1. d4 d5 2. c4**\n\nMain ideas:\n- fight for the center\n- open lines for White’s pieces\n- create pressure on Black’s d5 pawn\n\nThere are a few major responses by Black, like:\n- **Queen’s Gambit Accepted**: `2...dxc4`\n- **Queen’s Gambit Declined**: `2...e6`\n- **Slav Defense**: `2...c6`\n\nIf you want, I can also explain the difference between **Queen’s Gambit Accepted vs Declined**.'

In [25]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit"}
iteration #2...
function_call: search {"query":"queen's gambit chess opening"}
iteration #3...
function_call: search {"query":"queen gambit course faq"}
iteration #4...
ASSISTANT:
I couldn’t find any course FAQ entry about “queen gambit,” so I can’t answer it from the course materials. It may be off-topic for this course.

If you want, I can still help you with other course-related questions—anything else you’d like to explore?


'I couldn’t find any course FAQ entry about “queen gambit,” so I can’t answer it from the course materials. It may be off-topic for this course.\n\nIf you want, I can still help you with other course-related questions—anything else you’d like to explore?'

In [26]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [27]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [28]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [29]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [30]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [31]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [32]:
result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)

-> Response received


-> Response received


-> Response received


In [33]:
result.cost

CostInfo(input_cost=Decimal('0.00245625'), output_cost=Decimal('0.0013995'), total_cost=Decimal('0.00385575'))

In [34]:
result.all_messages

[EasyInputMessage(content="You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results \nand then perform more searches. \n\nThe question has to be about the course or its logistics, offtopic questions \nshouldn't be answered. If the search returns nothing, it's likely an off-topic question.\nIf you can't answer the question using FAQ, don't do it yourself. Only use the \nfacts from the FAQ database.\n\nAt the end, ask if there are other areas that the user wants to explore.", role='developer', phase=None, type=None),
 EasyInputMessage(content='How do I run Olama locally?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"Olama locally run local installation FAQ"}', call_id='call_97d

In [35]:
result2 = runner.loop(
    prompt="How do I run a different model?",
    previous_messages=result.all_messages,
    callback=callback,
)

-> Response received


-> Response received


In [ ]:
runner.run()